# Week 3 EDA

Goal: inspect MovieLens 25M raw data and identify concrete cleaning actions for the next notebook.

## 1) Setup

In [3]:
from pathlib import Path
import json
import polars as pl

RAW_DIR = Path('../data/raw/ml-25m')
INTERIM_DIR = Path('../data/interim')
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_DIR.exists():
    raise FileNotFoundError(f'Raw directory not found: {RAW_DIR}')

RAW_DIR

PosixPath('../data/raw/ml-25m')

## 2) Raw Files Inventory

In [4]:
raw_files = sorted([p for p in RAW_DIR.iterdir() if p.is_file()])
inventory = pl.DataFrame({
    'file': [p.name for p in raw_files],
    'size_mb': [round(p.stat().st_size / (1024**2), 2) for p in raw_files],
})
inventory.sort('size_mb', descending=True)

file,size_mb
str,f64
"""ratings.csv""",646.84
"""genome-scores.csv""",415.0
"""tags.csv""",37.01
"""movies.csv""",2.9
"""links.csv""",1.31
"""genome-tags.csv""",0.02
"""README.txt""",0.01


## 3) Table Shapes and Schema Samples

In [5]:
CORE_TABLES = {
    'movies': RAW_DIR / 'movies.csv',
    'ratings': RAW_DIR / 'ratings.csv',
    'tags': RAW_DIR / 'tags.csv',
    'links': RAW_DIR / 'links.csv',
}

OPTIONAL_TABLES = {
    'genome_scores': RAW_DIR / 'genome-scores.csv',
    'genome_tags': RAW_DIR / 'genome-tags.csv',
}

TABLES = {**CORE_TABLES, **OPTIONAL_TABLES}

shape_rows = []
schema_rows = []

for name, path in TABLES.items():
    lf = pl.scan_csv(path, infer_schema_length=10000)
    n_rows = lf.select(pl.len().alias('rows')).collect().item()
    sample = pl.read_csv(path, n_rows=2000, infer_schema_length=10000)

    shape_rows.append({
        'table': name,
        'rows': n_rows,
        'cols': sample.width,
        'size_mb': round(path.stat().st_size / (1024**2), 2),
        'scope': 'core' if name in CORE_TABLES else 'optional',
    })

    for col, dtype in sample.schema.items():
        schema_rows.append({
            'table': name,
            'column': col,
            'sample_dtype': str(dtype),
        })

shape_df = pl.DataFrame(shape_rows).sort('rows', descending=True)
schema_df = pl.DataFrame(schema_rows)
shape_df

table,rows,cols,size_mb,scope
str,i64,i64,f64,str
"""ratings""",25000095,4,646.84,"""core"""
"""genome_scores""",15584448,3,415.0,"""optional"""
"""tags""",1093360,4,37.01,"""core"""
"""movies""",62423,3,2.9,"""core"""
"""links""",62423,3,1.31,"""core"""
"""genome_tags""",1128,2,0.02,"""optional"""


In [6]:
table_usage_recommendation = pl.DataFrame([
    {'table': 'ratings', 'week3_use': 'required', 'reason': 'primary interaction layer and sparsity analysis'},
    {'table': 'movies', 'week3_use': 'required', 'reason': 'catalog layer and metadata base'},
    {'table': 'tags', 'week3_use': 'recommended', 'reason': 'text signal for future content and hybrid features'},
    {'table': 'links', 'week3_use': 'recommended', 'reason': 'external IDs and integration hooks'},
    {'table': 'genome_scores', 'week3_use': 'optional', 'reason': 'large feature source for week5+, can be deferred'},
    {'table': 'genome_tags', 'week3_use': 'optional', 'reason': 'lookup table for genome_scores tags'},
])

table_usage_recommendation

table,week3_use,reason
str,str,str
"""ratings""","""required""","""primary interaction layer and …"
"""movies""","""required""","""catalog layer and metadata bas…"
"""tags""","""recommended""","""text signal for future content…"
"""links""","""recommended""","""external IDs and integration h…"
"""genome_scores""","""optional""","""large feature source for week5…"
"""genome_tags""","""optional""","""lookup table for genome_scores…"


## 4) Missing Values by Column (All Tables)

In [7]:
missing_rows = []

for name, path in TABLES.items():
    lf = pl.scan_csv(path, infer_schema_length=10000)
    n_rows = lf.select(pl.len()).collect().item()
    cols = pl.read_csv(path, n_rows=5).columns

    null_counts = lf.select([pl.col(c).is_null().sum().alias(c) for c in cols]).collect()
    for c in cols:
        n_null = int(null_counts[0, c])
        missing_rows.append({
            'table': name,
            'column': c,
            'null_count': n_null,
            'null_pct': round((n_null / n_rows) * 100, 4) if n_rows else 0.0,
        })

missing_df = pl.DataFrame(missing_rows)
missing_df.filter(pl.col('null_count') > 0).sort(['null_pct', 'null_count'], descending=True)

table,column,null_count,null_pct
str,str,i64,f64
"""links""","""tmdbId""",107,0.1714


## 5) Integrity Checks Relevant for Cleaning

In [8]:
movies = pl.read_csv(TABLES['movies'], infer_schema_length=10000)
ratings = pl.read_csv(TABLES['ratings'], infer_schema_length=10000)
tags = pl.read_csv(TABLES['tags'], infer_schema_length=10000)
links = pl.read_csv(TABLES['links'], infer_schema_length=10000)
genome_scores = pl.read_csv(TABLES['genome_scores'], infer_schema_length=10000)
genome_tags = pl.read_csv(TABLES['genome_tags'], infer_schema_length=10000)

checks = {
    'ratings_missing_required': int(ratings.select(
        pl.any_horizontal([
            pl.col('userId').is_null(),
            pl.col('movieId').is_null(),
            pl.col('rating').is_null(),
            pl.col('timestamp').is_null(),
        ]).sum()
    ).item()),
    'ratings_out_of_range': int(ratings.filter((pl.col('rating') < 0.5) | (pl.col('rating') > 5.0)).height),
    'ratings_duplicate_user_movie_pairs': int(ratings.select(pl.struct(['userId', 'movieId']).is_duplicated().sum()).item()),
    'movies_duplicate_movieId': int(movies.select(pl.col('movieId').is_duplicated().sum()).item()),
    'movies_missing_title': int(movies.select(pl.col('title').is_null().sum()).item()),
    'movies_missing_genres': int(movies.select(pl.col('genres').is_null().sum()).item()),
    'tags_missing_tag_text': int(tags.select(pl.col('tag').is_null().sum()).item()),
    'tags_duplicate_full_rows': int(tags.is_duplicated().sum()),
    'links_missing_imdbId': int(links.select(pl.col('imdbId').is_null().sum()).item()),
    'links_missing_tmdbId': int(links.select(pl.col('tmdbId').is_null().sum()).item()),
    'links_duplicate_movieId': int(links.select(pl.col('movieId').is_duplicated().sum()).item()),
    'genome_scores_duplicate_movie_tag': int(genome_scores.select(pl.struct(['movieId', 'tagId']).is_duplicated().sum()).item()),
    'genome_tags_duplicate_tagId': int(genome_tags.select(pl.col('tagId').is_duplicated().sum()).item()),
}

pl.DataFrame({
    'check': list(checks.keys()),
    'value': list(checks.values()),
}).sort('value', descending=True)

check,value
str,i64
"""links_missing_tmdbId""",107
"""ratings_missing_required""",0
"""ratings_out_of_range""",0
"""ratings_duplicate_user_movie_p…",0
"""movies_duplicate_movieId""",0
…,…
"""tags_duplicate_full_rows""",0
"""links_missing_imdbId""",0
"""links_duplicate_movieId""",0


In [9]:
movies_ids = movies.select('movieId').unique()
ratings_ids = ratings.select('movieId').unique()
tags_ids = tags.select('movieId').unique()
genome_movie_ids = genome_scores.select('movieId').unique()
genome_tag_ids = genome_scores.select('tagId').unique()

orphan_ratings_movie_ids = ratings_ids.join(movies_ids, on='movieId', how='anti').height
orphan_tags_movie_ids = tags_ids.join(movies_ids, on='movieId', how='anti').height
orphan_genome_movie_ids = genome_movie_ids.join(movies_ids, on='movieId', how='anti').height
orphan_genome_tag_ids = genome_tag_ids.join(genome_tags.select('tagId').unique(), on='tagId', how='anti').height

join_coverage_df = pl.DataFrame({
    'relationship_check': [
        'ratings movieId matched in movies',
        'tags movieId matched in movies',
        'genome_scores movieId matched in movies',
        'genome_scores tagId matched in genome_tags',
    ],
    'unmatched_count': [
        int(orphan_ratings_movie_ids),
        int(orphan_tags_movie_ids),
        int(orphan_genome_movie_ids),
        int(orphan_genome_tag_ids),
    ],
    'coverage_pct': [
        round((1 - (orphan_ratings_movie_ids / max(1, ratings_ids.height))) * 100, 4),
        round((1 - (orphan_tags_movie_ids / max(1, tags_ids.height))) * 100, 4),
        round((1 - (orphan_genome_movie_ids / max(1, genome_movie_ids.height))) * 100, 4),
        round((1 - (orphan_genome_tag_ids / max(1, genome_tag_ids.height))) * 100, 4),
    ],
})

join_coverage_df

relationship_check,unmatched_count,coverage_pct
str,i64,f64
"""ratings movieId matched in mov…",0,100.0
"""tags movieId matched in movies""",0,100.0
"""genome_scores movieId matched …",0,100.0
"""genome_scores tagId matched in…",0,100.0


## 6) Scale and Sparsity Metrics for Week 3 Report

In [10]:
users_n = ratings.select(pl.col('userId').n_unique()).item()
items_n = ratings.select(pl.col('movieId').n_unique()).item()
interactions_n = ratings.height
possible_interactions = int(users_n * items_n)

sparsity = 1 - (interactions_n / possible_interactions)

scale_metrics = {
    'users': int(users_n),
    'items': int(items_n),
    'interactions': int(interactions_n),
    'possible_user_item_pairs': possible_interactions,
    'density_pct': round((interactions_n / possible_interactions) * 100, 6),
    'sparsity_pct': round(sparsity * 100, 6),
}

table_memory_df = shape_df.select(['table', 'rows', 'cols', 'size_mb', 'scope']).sort('size_mb', descending=True)

scale_metrics, table_memory_df

({'users': 162541,
  'items': 59047,
  'interactions': 25000095,
  'possible_user_item_pairs': 9597558427,
  'density_pct': 0.260484,
  'sparsity_pct': 99.739516},
 shape: (6, 5)
 ┌───────────────┬──────────┬──────┬─────────┬──────────┐
 │ table         ┆ rows     ┆ cols ┆ size_mb ┆ scope    │
 │ ---           ┆ ---      ┆ ---  ┆ ---     ┆ ---      │
 │ str           ┆ i64      ┆ i64  ┆ f64     ┆ str      │
 ╞═══════════════╪══════════╪══════╪═════════╪══════════╡
 │ ratings       ┆ 25000095 ┆ 4    ┆ 646.84  ┆ core     │
 │ genome_scores ┆ 15584448 ┆ 3    ┆ 415.0   ┆ optional │
 │ tags          ┆ 1093360  ┆ 4    ┆ 37.01   ┆ core     │
 │ movies        ┆ 62423    ┆ 3    ┆ 2.9     ┆ core     │
 │ links         ┆ 62423    ┆ 3    ┆ 1.31    ┆ core     │
 │ genome_tags   ┆ 1128     ┆ 2    ┆ 0.02    ┆ optional │
 └───────────────┴──────────┴──────┴─────────┴──────────┘)

## 7) Quick Distribution Checks

In [11]:
rating_dist = ratings.group_by('rating').len().sort('rating')
rating_dist

rating,len
f64,u32
0.5,393068
1.0,776815
1.5,399490
2.0,1640868
2.5,1262797
3.0,4896928
3.5,3177318
4.0,6639798
4.5,2200539


In [12]:
top_genres = (
    movies
    .with_columns(pl.col('genres').str.split('|'))
    .explode('genres')
    .group_by('genres')
    .len()
    .sort('len', descending=True)
    .head(15)
)
top_genres

genres,len
str,u32
"""Drama""",25606
"""Comedy""",16870
"""Thriller""",8654
"""Romance""",7719
"""Action""",7348
…,…
"""Sci-Fi""",3595
"""Children""",2935
"""Animation""",2929


In [13]:
ratings_time = ratings.select(
    pl.from_epoch('timestamp', time_unit='s').alias('rated_at')
)

ratings_time.select([
    pl.col('rated_at').min().alias('min_timestamp'),
    pl.col('rated_at').max().alias('max_timestamp'),
])

min_timestamp,max_timestamp
datetime[μs],datetime[μs]
1995-01-09 11:46:49,2019-11-21 09:15:03


## 8) Export EDA Summary for Cleaning and Week 3 Docs

In [14]:
summary = {
    'table_shapes': shape_df.to_dicts(),
    'recommended_table_usage': table_usage_recommendation.to_dicts(),
    'missing_columns_nonzero': missing_df.filter(pl.col('null_count') > 0).sort(['table', 'null_count'], descending=True).to_dicts(),
    'quality_checks': checks,
    'join_coverage': join_coverage_df.to_dicts(),
    'scale_metrics': scale_metrics,
    'table_memory_estimate_mb': table_memory_df.to_dicts(),
    'rating_distribution': rating_dist.to_dicts(),
    'top_genres': top_genres.to_dicts(),
}

summary_path = INTERIM_DIR / 'week03_eda_summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

scale_path = INTERIM_DIR / 'week03_scale_metrics.json'
scale_path.write_text(json.dumps({
    'scale_metrics': scale_metrics,
    'table_memory_estimate_mb': table_memory_df.to_dicts(),
}, indent=2), encoding='utf-8')

dictionary_profile_df = (
    schema_df
    .join(
        missing_df.select(['table', 'column', 'null_count', 'null_pct']),
        on=['table', 'column'],
        how='left',
    )
    .with_columns([
        pl.col('null_count').fill_null(0),
        pl.col('null_pct').fill_null(0.0),
    ])
    .sort(['table', 'column'])
)

dictionary_profile_df.write_csv(INTERIM_DIR / 'week03_dictionary_profile.csv')

summary_path, scale_path, INTERIM_DIR / 'week03_dictionary_profile.csv'

(PosixPath('../data/interim/week03_eda_summary.json'),
 PosixPath('../data/interim/week03_scale_metrics.json'),
 PosixPath('../data/interim/week03_dictionary_profile.csv'))

## 9) Cleaning Decisions to Implement Next

Use the outputs above to define your cleaning notebook rules, for example:
- keep `ratings` and `movies` as required Week 3 core tables
- keep `tags` and `links` if useful for immediate processed V1 and metadata joins
- defer `genome_scores` feature-heavy processing to Week 5 if runtime is high
- preserve rows with null `tmdbId` (107 rows) but document this in data dictionary
- persist cleaned V1 tables to `data/processed/` with explicit key constraints
- document which tables are in-scope now vs deferred, and why